# BASELINE 2: RAWNET2 (END-TO-END AUDIO ANTI-SPOOFING)
**Arsitektur:** RawNet2 Murni (End-to-End dari Sinyal Audio Mentah)  
**Peran:** Sebagai pembanding model *State-of-the-Art* (SOTA) yang tidak menggunakan ekstraksi fitur (*featureless*), untuk membuktikan bahwa tanpa fusi fitur khusus, model akan kesulitan melakukan generalisasi pada data baru.

Notebook ini diorganisasikan ke dalam beberapa bagian berikut:
1. **Determinisme & Pengaturan Random Seed:** Mengatur seed acak agar hasil eksperimen dapat diulang secara deterministik.
2. **Tempat Import Dataset & Dataloader (ASVspoof 2019 LA):** Tempat khusus bagi pengguna untuk memuat dataset lokal.
3. **Prapemrosesan Sinyal Audio Mentah:** Logika pemotongan (*truncating*) dan pengulangan (*tiling/padding*) audio agar berukuran seragam (64.000 sampel).
4. **Pendefinisian Model RawNet2 (Standalone):** Pendefinisian arsitektur RawNet2 secara mandiri di dalam notebook agar bisa dijalankan di Kaggle atau Colab tanpa membutuhkan berkas eksternal.
5. **Cross-Dataset Evaluation (ASVspoof 5):** Memuat dataset streaming ASVspoof 5 dari Hugging Face dan mengimplementasikan *Stratified Sampling* untuk mengambil subset berimbang (maksimal 5.000 sampel).
6. **Metrik Evaluasi:** Perhitungan EER dan min t-DCF.
7. **Training & Validation Loop:** Loop pelatihan penuh menggunakan Adam optimizer.

## 1. Determinisme & Pengaturan Random Seed
Agar hasil eksperimen bersifat deterministik, kita mengunci seed acak pada Python, NumPy, dan PyTorch.

In [ ]:
import os
import random
import numpy as np
import torch

def set_seed(seed=1234):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(1234)
print("Determinisme aktif. Random seed diatur ke 1234 (sesuai konfigurasi default RawNet2).")

## 2. Tempat Import Dataset (Dataloader)
> **Catatan Penting:** Gunakan sel di bawah ini untuk memuat dataset audio Anda (misalnya ASVspoof 2019 LA).
> Kelas Dataset di bawah ini didesain khusus untuk menerima berkas audio mentah (.wav atau .flac).

In [ ]:
# =====================================================================
# TEMPAT UNTUK IMPORT DATASET & INISIALISASI DATALOADER (Mendukung ASVspoof 5 & 2019)
# =====================================================================
from torch.utils.data import Dataset, DataLoader
import torchaudio
import pandas as pd
import numpy as np
import os
import random
import torch

class RawAudioASVDataset(Dataset):
    """
    Dataset PyTorch untuk Audio Anti-Spoofing.
    Mendukung ASVspoof 2019 & ASVspoof 5 dengan pemindaian subfolder secara rekursif.
    Menerapkan Stratified Quota Sampling pada ASVspoof 5 untuk optimasi RAM.
    """
    def __init__(self, protocols_file, wav_dir, max_len=64000, transform=None):
        self.wav_dir = wav_dir
        self.max_len = max_len
        self.transform = transform
        self.file_list = []
        self.labels = []
        
        # Membaca berkas protokol
        if protocols_file and os.path.exists(protocols_file):
            print(f"Membaca protokol dari {protocols_file}...")
            try:
                # Cek baris pertama untuk mendeteksi tipe protokol
                with open(protocols_file, 'r', encoding='utf-8') as f:
                    first_line = f.readline().strip()
                n_cols = len(first_line.split())
                
                if n_cols >= 9: # ASVspoof 5 (10 kolom)
                    print("Mendeteksi format protokol ASVspoof 5. Menerapkan Stratified Quota Sampling...")
                    columns = [
                        "SPEAKER_ID", "FLAC_FILE_NAME", "SPEAKER_GENDER", "CODEC", "CODEC_Q",
                        "CODEC_SEED", "ATTACK_TAG", "ATTACK_LABEL", "KEY", "TMP"
                    ]
                    df = pd.read_csv(protocols_file, sep=r'\s+', header=None, names=columns[:n_cols])
                    
                    # Quota sampling (1000 bona fide, dan spoof disampling seimbang per jenis serangan)
                    max_bonafide = 1000
                    df_bonafide = df[df['KEY'] == 'bonafide']
                    df_spoof = df[df['KEY'] == 'spoof']
                    
                    n_bonafide = min(len(df_bonafide), max_bonafide)
                    df_bonafide_sampled = df_bonafide.sample(n=n_bonafide, random_state=42)
                    
                    df_spoof_sampled = pd.DataFrame()
                    if len(df_spoof) > 0:
                        attack_groups = df_spoof.groupby('ATTACK_LABEL')
                        n_groups = len(attack_groups)
                        samples_per_group = max_bonafide // n_groups
                        
                        for label, group in attack_groups:
                            n_take = min(len(group), samples_per_group)
                            df_grp_sampled = group.sample(n=n_take, random_state=42)
                            df_spoof_sampled = pd.concat([df_spoof_sampled, df_grp_sampled])
                            
                        # Penuhi sisa kuota jika ada group yang kurang sampelnya
                        current_collected = len(df_spoof_sampled)
                        if current_collected < max_bonafide:
                            needed = max_bonafide - current_collected
                            remaining_spoof = df_spoof[~df_spoof.index.isin(df_spoof_sampled.index)]
                            if len(remaining_spoof) > 0:
                                n_take = min(len(remaining_spoof), needed)
                                extra_samples = remaining_spoof.sample(n=n_take, random_state=42)
                                df_spoof_sampled = pd.concat([df_spoof_sampled, extra_samples])
                                
                    df_final = pd.concat([df_bonafide_sampled, df_spoof_sampled])
                    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
                    
                    self.file_list = df_final['FLAC_FILE_NAME'].tolist()
                    self.labels = [1 if k == 'bonafide' else 0 for k in df_final['KEY']]
                    print(f"Berhasil memuat {len(self.file_list)} sampel terstratifikasi ASVspoof 5.")
                else: # ASVspoof 2019 (5 kolom)
                    with open(protocols_file, 'r', encoding='utf-8') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) >= 5:
                                file_name = parts[1]
                                key = parts[4]
                                label = 1 if key == 'bonafide' else 0
                                self.file_list.append(file_name)
                                self.labels.append(label)
                    print(f"Berhasil memuat {len(self.file_list)} entri dataset ASVspoof 2019.")
            except Exception as e:
                print(f"Error membaca berkas protokol: {e}")
                self.file_list = []
                self.labels = []
        else:
            print(f"PERINGATAN: Berkas protokol tidak ditemukan di '{protocols_file}'.")
            print("Membuat DUMMY dataset untuk pengujian...")
            self.file_list = [f"dummy_sample_{i}" for i in range(100)]
            self.labels = [random.choice([0, 1]) for _ in range(100)]

        # Bangun indeks path file audio secara rekursif untuk mendukung folder bersarang (flac_T_aa, dll.)
        self.path_map = {}
        if os.path.exists(wav_dir):
            print(f"Membangun indeks path audio secara rekursif di {wav_dir}...")
            for root, _, files in os.walk(wav_dir):
                for f in files:
                    if f.endswith(('.flac', '.wav')):
                        name_without_ext = os.path.splitext(f)[0]
                        self.path_map[name_without_ext] = os.path.join(root, f)
            print(f"Selesai! Berhasil mengindeks {len(self.path_map)} file audio.")
        else:
            print(f"PERINGATAN: Direktori audio '{wav_dir}' tidak ditemukan.")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        label = self.labels[idx]
        
        # Cari path dari indeks map
        wav_path = self.path_map.get(file_name)
        if not wav_path:
            wav_path = os.path.join(self.wav_dir, f"{file_name}.flac")
            if not os.path.exists(wav_path):
                wav_path = os.path.join(self.wav_dir, f"{file_name}.wav")
                
        if wav_path and os.path.exists(wav_path):
            waveform, sr = torchaudio.load(wav_path)
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                waveform = resampler(waveform)
            waveform = waveform.squeeze(0)
        else:
            waveform = torch.randn(self.max_len) * 0.01
            
        # Padding atau potong agar ukuran waveform seragam
        if waveform.shape[0] < self.max_len:
            pad_len = self.max_len - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        else:
            waveform = waveform[:self.max_len]
            
        # Terapkan transformasi (ekstraksi fitur) jika disediakan
        if self.transform:
            feature = self.transform(waveform)
            return feature, label
            
        return torch.FloatTensor(waveform), label

# SILAKAN EDIT PATH BASE DIREKTORI DI BAWAH INI SESUAI DENGAN LOKASI DATASET ASVSPOOF 5 LOKAL ANDA
BASE_DIR = "D:/Lomba/Gemastik 2026/Data Mining/panduan/Dataset_ASVspoof5_Sampled"
PATH_PROTOKOL_TRAIN = os.path.join(BASE_DIR, "ASVspoof5.train.tsv")
PATH_WAV_DIR_TRAIN = BASE_DIR

PATH_PROTOKOL_DEV = os.path.join(BASE_DIR, "ASVspoof5.dev.track_1.tsv")
PATH_WAV_DIR_DEV = BASE_DIR

print("Dataset loader ASVspoof 5 lokal siap digunakan.")


## 3. Logika Pemrosesan Sinyal Audio Mentah (End-to-End)
Dalam pemrosesan audio mentah untuk RawNet2:
- **Mengapa tidak menggunakan MFCC/LFCC?** RawNet2 adalah model *featureless* yang menggunakan filter SincConv sebagai lapisan pertama untuk mengekstrak filterbank secara adaptif dari sinyal domain waktu.
- **Logika Pemotongan/Pengulangan:**
  - **Truncating (Pemotongan):** Jika durasi audio melebihi batas waktu 4 detik (64.000 sampel), audio dipotong karena informasi antispoofing biasanya tersebar di seluruh durasi dan 4 detik sudah memadai untuk ekstraksi fitur temporal.
  - **Tiling/Repeating (Pengulangan):** Jika audio lebih pendek dari 4 detik, audio diulang menggunakan `np.tile(x, num_repeats)`. Strategi pengulangan lebih disukai daripada *zero-padding* karena zero-padding memperkenalkan diskontinuitas buatan (frekuensi tinggi artifisial pada ujung batas nol) yang dapat mengganggu filter SincConv adaptif.

## 4. Pendefinisian Model RawNet2 (Standalone)
Di sini kita mendefinisikan arsitektur **RawNet2** secara langsung di dalam notebook agar berkas `.ipynb` ini mandiri (*self-contained*) dan siap dijalankan di Kaggle, Colab, atau platform cloud lainnya.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# LAPISAN SINC-CONVOLUTION (SincConv) DARI RAWNET2
# =====================================================================
class SincConv(nn.Module):
    @staticmethod
    def to_mel(hz):
        return 2595 * np.log10(1 + hz / 700)

    @staticmethod
    def to_hz(mel):
        return 700 * (10 ** (mel / 2595) - 1)

    def __init__(self, device, out_channels, kernel_size, in_channels=1, sample_rate=16000,
                 stride=1, padding=0, dilation=1, bias=False, groups=1, freq_scale='Mel'):
        super(SincConv, self).__init__()
        if in_channels != 1:
            msg = "SincConv only support one input channel (here, in_channels = {%i})" % (in_channels)
            raise ValueError(msg)
        
        self.out_channels = out_channels + 1
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate
        if kernel_size % 2 == 0:
            self.kernel_size = self.kernel_size + 1

        self.device = device   
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        
        if bias:
            raise ValueError('SincConv does not support bias.')
        if groups > 1:
            raise ValueError('SincConv does not support groups.')
        
        NFFT = 512
        f = int(self.sample_rate / 2) * np.linspace(0, 1, int(NFFT / 2) + 1)

        if freq_scale == 'Mel':
            fmel = self.to_mel(f)
            fmelmax = np.max(fmel)
            fmelmin = np.min(fmel)
            filbandwidthsmel = np.linspace(fmelmin, fmelmax, self.out_channels + 2)
            filbandwidthsf = self.to_hz(filbandwidthsmel)
            self.freq = filbandwidthsf[:self.out_channels]
        elif freq_scale == 'Inverse-mel':
            fmel = self.to_mel(f)
            fmelmax = np.max(fmel)
            fmelmin = np.min(fmel)
            filbandwidthsmel = np.linspace(fmelmin, fmelmax, self.out_channels + 2)
            filbandwidthsf = self.to_hz(filbandwidthsmel)
            self.mel = filbandwidthsf[:self.out_channels]
            self.freq = np.abs(np.flip(self.mel) - 1)
        else:
            fmelmax = np.max(f)
            fmelmin = np.min(f)
            filbandwidthsmel = np.linspace(fmelmin, fmelmax, self.out_channels + 2)
            self.freq = filbandwidthsmel[:self.out_channels]
        
        self.hsupp = torch.arange(-(self.kernel_size - 1) / 2, (self.kernel_size - 1) / 2 + 1)
        self.band_pass = torch.zeros(self.out_channels - 1, self.kernel_size)
        
    def forward(self, x):
        for i in range(len(self.freq) - 1):
            fmin = self.freq[i]
            fmax = self.freq[i + 1]
            hHigh = (2 * fmax / self.sample_rate) * np.sinc(2 * fmax * self.hsupp / self.sample_rate)
            hLow = (2 * fmin / self.sample_rate) * np.sinc(2 * fmin * self.hsupp / self.sample_rate)
            hideal = hHigh - hLow
            self.band_pass[i, :] = torch.Tensor(np.hamming(self.kernel_size)) * torch.Tensor(hideal)
        
        band_pass_filter = self.band_pass.to(self.device)
        self.filters = (band_pass_filter).view(self.out_channels - 1, 1, self.kernel_size)
        return F.conv1d(x, self.filters, stride=self.stride, padding=self.padding, dilation=self.dilation, bias=None, groups=1)

# =====================================================================
# RESIDUAL BLOCK DARI RAWNET2
# =====================================================================
class Residual_block(nn.Module):
    def __init__(self, nb_filts, first=False):
        super(Residual_block, self).__init__()
        self.first = first
        if not self.first:
            self.bn1 = nn.BatchNorm1d(num_features=nb_filts[0])
        self.lrelu = nn.LeakyReLU(negative_slope=0.3)
        self.conv1 = nn.Conv1d(in_channels=nb_filts[0], out_channels=nb_filts[1], kernel_size=3, padding=1, stride=1)
        self.bn2 = nn.BatchNorm1d(num_features=nb_filts[1])
        self.conv2 = nn.Conv1d(in_channels=nb_filts[1], out_channels=nb_filts[1], padding=1, kernel_size=3, stride=1)
        
        if nb_filts[0] != nb_filts[1]:
            self.downsample = True
            self.conv_downsample = nn.Conv1d(in_channels=nb_filts[0], out_channels=nb_filts[1], padding=0, kernel_size=1, stride=1)
        else:
            self.downsample = False
        self.mp = nn.MaxPool1d(3)
        
    def forward(self, x):
        identity = x
        if not self.first:
            out = self.bn1(x)
            out = self.lrelu(out)
        else:
            out = x
            
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.lrelu(out)
        out = self.conv2(out)
        
        if self.downsample:
            identity = self.conv_downsample(identity)
            
        out += identity
        out = self.mp(out)
        return out

# =====================================================================
# ARSITEKTUR UTAMA RAWNET2
# =====================================================================
class RawNet(nn.Module):
    def __init__(self, d_args, device):
        super(RawNet, self).__init__()
        self.device = device
        self.Sinc_conv = SincConv(
            device=self.device,
            out_channels=d_args['filts'][0],
            kernel_size=d_args['first_conv'],
            in_channels=d_args['in_channels'],
            freq_scale='Mel'
        )
        self.first_bn = nn.BatchNorm1d(num_features=d_args['filts'][0])
        self.selu = nn.SELU(inplace=True)
        self.block0 = nn.Sequential(Residual_block(nb_filts=d_args['filts'][1], first=True))
        self.block1 = nn.Sequential(Residual_block(nb_filts=d_args['filts'][1]))
        self.block2 = nn.Sequential(Residual_block(nb_filts=d_args['filts'][2]))
        
        # Salin list filts[2] agar tidak merusak config eksternal
        filts_2 = list(d_args['filts'][2])
        filts_2[0] = filts_2[1]
        self.block3 = nn.Sequential(Residual_block(nb_filts=filts_2))
        self.block4 = nn.Sequential(Residual_block(nb_filts=filts_2))
        self.block5 = nn.Sequential(Residual_block(nb_filts=filts_2))
        self.avgpool = nn.AdaptiveAvgPool1d(1)

        self.fc_attention0 = self._make_attention_fc(in_features=d_args['filts'][1][-1], l_out_features=d_args['filts'][1][-1])
        self.fc_attention1 = self._make_attention_fc(in_features=d_args['filts'][1][-1], l_out_features=d_args['filts'][1][-1])
        self.fc_attention2 = self._make_attention_fc(in_features=d_args['filts'][2][-1], l_out_features=d_args['filts'][2][-1])
        self.fc_attention3 = self._make_attention_fc(in_features=d_args['filts'][2][-1], l_out_features=d_args['filts'][2][-1])
        self.fc_attention4 = self._make_attention_fc(in_features=d_args['filts'][2][-1], l_out_features=d_args['filts'][2][-1])
        self.fc_attention5 = self._make_attention_fc(in_features=d_args['filts'][2][-1], l_out_features=d_args['filts'][2][-1])

        self.bn_before_gru = nn.BatchNorm1d(num_features=d_args['filts'][2][-1])
        self.gru = nn.GRU(
            input_size=d_args['filts'][2][-1],
            hidden_size=d_args['gru_node'],
            num_layers=d_args['nb_gru_layer'],
            batch_first=True
        )
        
        self.fc1_gru = nn.Linear(in_features=d_args['gru_node'], out_features=d_args['nb_fc_node'])
        self.fc2_gru = nn.Linear(in_features=d_args['nb_fc_node'], out_features=d_args['nb_classes'], bias=True)
        self.sig = nn.Sigmoid()
        
    def forward(self, x, y=None, is_test=False):
        nb_samp = x.shape[0]
        len_seq = x.shape[1]
        x = x.view(nb_samp, 1, len_seq)
        
        x = self.Sinc_conv(x)
        x = F.max_pool1d(torch.abs(x), 3)
        x = self.first_bn(x)
        x = self.selu(x)
        
        x0 = self.block0(x)
        y0 = self.avgpool(x0).view(x0.size(0), -1)
        y0 = self.fc_attention0(y0)
        y0 = self.sig(y0).view(y0.size(0), y0.size(1), -1)
        x = x0 * y0 + y0
        
        x1 = self.block1(x)
        y1 = self.avgpool(x1).view(x1.size(0), -1)
        y1 = self.fc_attention1(y1)
        y1 = self.sig(y1).view(y1.size(0), y1.size(1), -1)
        x = x1 * y1 + y1

        x2 = self.block2(x)
        y2 = self.avgpool(x2).view(x2.size(0), -1)
        y2 = self.fc_attention2(y2)
        y2 = self.sig(y2).view(y2.size(0), y2.size(1), -1)
        x = x2 * y2 + y2

        x3 = self.block3(x)
        y3 = self.avgpool(x3).view(x3.size(0), -1)
        y3 = self.fc_attention3(y3)
        y3 = self.sig(y3).view(y3.size(0), y3.size(1), -1)
        x = x3 * y3 + y3

        x4 = self.block4(x)
        y4 = self.avgpool(x4).view(x4.size(0), -1)
        y4 = self.fc_attention4(y4)
        y4 = self.sig(y4).view(y4.size(0), y4.size(1), -1)
        x = x4 * y4 + y4

        x5 = self.block5(x)
        y5 = self.avgpool(x5).view(x5.size(0), -1)
        y5 = self.fc_attention5(y5)
        y5 = self.sig(y5).view(y5.size(0), y5.size(1), -1)
        x = x5 * y5 + y5

        x = self.bn_before_gru(x)
        x = self.selu(x)
        x = x.permute(0, 2, 1)
        self.gru.flatten_parameters()
        x, _ = self.gru(x)
        x = x[:, -1, :]
        x = self.fc1_gru(x)
        x = self.fc2_gru(x)

        if not is_test:
            output = x
            return output
        else:
            output = F.softmax(x, dim=1)
            return output

    def _make_attention_fc(self, in_features, l_out_features):
        l_fc = []
        l_fc.append(nn.Linear(in_features=in_features, out_features=l_out_features))
        return nn.Sequential(*l_fc)

# Konfigurasi parameter RawNet2
rawnet2_config = {
    'first_conv': 128,
    'in_channels': 1,
    'filts': [128, [128, 128], [128, 512], [512, 512]],
    'nb_fc_node': 1024,
    'gru_node': 1024,
    'nb_gru_layer': 3,
    'nb_classes': 2
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RawNet(rawnet2_config, device).to(device)
print(f"Inisialisasi model RawNet selesai di perangkat: {device} (Standalone Mode)")

## 5. Cross-Dataset Evaluation (ASVspoof 5) & Stratified Sampling
Untuk membuktikan batas performa model (*lower-bound baseline*) dan kemampuan generalisasinya pada data baru, kita melakukan evaluasi lintas-dataset (*cross-dataset*) pada dataset **ASVspoof 5** menggunakan pustaka Hugging Face `datasets`.

### Mengapa menggunakan Stratified Sampling?
1. **Penyeimbangan Kelas:** Dataset audio spoofing sering kali tidak seimbang, di mana jumlah berkas spoof jauh lebih banyak dibanding bona fide karena banyaknya variasi metode sintesis/konversi suara. Stratified sampling memastikan proporsi *Bona fide* (50%) dan *Spoof* (50%) tetap seimbang.
2. **Keterwakilan Attack ID secara Merata:** ASVspoof 5 memiliki beragam algoritma spoofing (*Attack ID* atau *system ID*). Stratified sampling memastikan sampel spoof yang diambil terbagi rata untuk masing-masing *Attack ID* yang ada, sehingga performa model dinilai secara adil terhadap semua jenis serangan, bukan hanya didominasi oleh jenis serangan tertentu yang jumlah sampelnya melimpah.
3. **Efisiensi Streaming:** Dataset ASVspoof 5 berukuran sangat besar. Mengunduh seluruh dataset akan memakan waktu lama dan menghabiskan ruang penyimpanan. Dengan mengaktifkan `streaming=True` dan menghentikan pengumpulan setelah kuota 5.000 sampel terstratifikasi terpenuhi, proses evaluasi menjadi sangat cepat dan efisien.

In [ ]:
try:
    import datasets
except ImportError:
    print("Menginstall pustaka datasets...")
    subprocess.run(["pip", "install", "datasets", "-q"])
    import datasets

from datasets import load_dataset
import pandas as pd
import numpy as np

def collect_stratified_asvspoof5_tsv(tsv_path, max_samples=5000):
    """
    Membaca berkas protokol TSV ASVspoof 5 secara lokal,
    melakukan stratified sampling:
    - Bona fide: 50%, Spoof: 50%
    - Spoof tersampel secara merata berdasarkan Attack ID (ATTACK_LABEL)
    """
    columns = [
        "SPEAKER_ID", "FLAC_FILE_NAME", "SPEAKER_GENDER", "CODEC", "CODEC_Q",
        "CODEC_SEED", "ATTACK_TAG", "ATTACK_LABEL", "KEY", "TMP"
    ]
    print(f"Membaca berkas protokol dari {tsv_path}...")
    df = pd.read_csv(tsv_path, sep=r'\s+', header=None, names=columns)
    print(f"Total baris ditemukan: {len(df)}")
    
    target_class_samples = max_samples // 2
    
    # Pisahkan bona fide dan spoof
    df_bonafide = df[df['KEY'] == 'bonafide']
    df_spoof = df[df['KEY'] == 'spoof']
    
    # Ambil sampel bona fide
    n_bonafide = min(len(df_bonafide), target_class_samples)
    df_bonafide_sampled = df_bonafide.sample(n=n_bonafide, random_state=42)
    
    # Stratifikasi sampel spoof berdasarkan ATTACK_LABEL
    df_spoof_sampled = pd.DataFrame()
    if len(df_spoof) > 0:
        attack_groups = df_spoof.groupby('ATTACK_LABEL')
        n_groups = len(attack_groups)
        samples_per_group = target_class_samples // n_groups
        
        # Ambil sampel dari setiap group secara merata
        for label, group in attack_groups:
            n_take = min(len(group), samples_per_group)
            df_grp_sampled = group.sample(n=n_take, random_state=42)
            df_spoof_sampled = pd.concat([df_spoof_sampled, df_grp_sampled])
            
        current_collected = len(df_spoof_sampled)
        if current_collected < target_class_samples:
            needed = target_class_samples - current_collected
            remaining_spoof = df_spoof[~df_spoof.index.isin(df_spoof_sampled.index)]
            if len(remaining_spoof) > 0:
                n_take = min(len(remaining_spoof), needed)
                extra_samples = remaining_spoof.sample(n=n_take, random_state=42)
                df_spoof_sampled = pd.concat([df_spoof_sampled, extra_samples])
                
    # Gabungkan
    df_final = pd.concat([df_bonafide_sampled, df_spoof_sampled])
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    samples_list = []
    for _, row in df_final.iterrows():
        samples_list.append({
            'file_name': row['FLAC_FILE_NAME'],
            'label': 1 if row['KEY'] == 'bonafide' else 0,
            'system_id': row['ATTACK_LABEL']
        })
        
    print(f"Selesai! Mengumpulkan {len(df_bonafide_sampled)} Bona fide dan {len(df_spoof_sampled)} Spoof.")
    return samples_list

def collect_stratified_asvspoof5(dataset_stream, max_samples=5000):
    """
    Mengambil sampel terstratifikasi secara round-robin dari dataset streaming ASVspoof 5 (Hugging Face fallback).
    """
    target_class_samples = max_samples // 2
    bona_fide_samples = []
    spoof_by_attack = {}
    
    print("Mengekstrak sampel dari streaming dataset... Mohon tunggu sebentar.")
    
    for item in dataset_stream:
        label = item.get('label')
        if label is None:
            label = item.get('key') or item.get('class')
            
        is_bonafide = False
        if label in [1, '1', 'bonafide', 'bona_fide', 'human']:
            is_bonafide = True
        elif label in [0, '0', 'spoof', 'spoofed', 'attack']:
            is_bonafide = False
        else:
            is_bonafide = 'bonafide' in str(label).lower() or 'human' in str(label).lower()
            
        if is_bonafide:
            if len(bona_fide_samples) < target_class_samples:
                bona_fide_samples.append(item)
        else:
            attack_id = item.get('attack_type') or item.get('attack_id') or item.get('system_id') or item.get('attack') or 'unknown'
            if attack_id not in spoof_by_attack:
                spoof_by_attack[attack_id] = []
            spoof_by_attack[attack_id].append(item)
            
        total_spoof_collected = sum(len(v) for v in spoof_by_attack.values())
        if len(bona_fide_samples) >= target_class_samples and total_spoof_collected >= target_class_samples * 2:
            break
            
    spoof_samples = []
    if spoof_by_attack:
        attack_ids = list(spoof_by_attack.keys())
        idx = 0
        while len(spoof_samples) < target_class_samples:
            added = False
            for aid in attack_ids:
                if idx < len(spoof_by_attack[aid]):
                    spoof_samples.append(spoof_by_attack[aid][idx])
                    added = True
                    if len(spoof_samples) >= target_class_samples:
                        break
            if not added:
                break
            idx += 1
            
    final_samples = bona_fide_samples + spoof_samples
    random.shuffle(final_samples)
    print(f"Selesai! Mengumpulkan {len(bona_fide_samples)} Bona fide dan {len(spoof_samples)} Spoof.")
    return final_samples

# Jalur lokal untuk berkas manifes dan direktori audio
PATH_ASV5_TSV = "D:/Lomba/Gemastik 2026/Data Mining/panduan/Dataset_ASVspoof5_Sampled/ASVspoof5.dev.track_1.tsv"
if not os.path.exists(PATH_ASV5_TSV):
    PATH_ASV5_TSV = "../../Dataset_ASVspoof5_Sampled/ASVspoof5.dev.track_1.tsv"
    
PATH_ASV5_WAV_DIR = "D:/Lomba/Gemastik 2026/Data Mining/panduan/dataset/asvspoof5/dev/flac"

try:
    if os.path.exists(PATH_ASV5_TSV):
        print(f"Memuat dataset dari berkas TSV lokal: {PATH_ASV5_TSV}")
        eval_samples = collect_stratified_asvspoof5_tsv(PATH_ASV5_TSV, max_samples=5000)
        use_local_asv5 = True
    else:
        print("Berkas TSV lokal tidak ditemukan. Menggunakan streaming dari Hugging Face...")
        asv5_dataset = load_dataset("jungjee/asvspoof5", streaming=True)
        stream_split = asv5_dataset['train']
        eval_samples = collect_stratified_asvspoof5(stream_split, max_samples=5000)
        use_local_asv5 = False
except Exception as e:
    print(f"Error memuat dataset ASVspoof 5: {e}")
    raise e


## 6. Metrik Evaluasi: Equal Error Rate (EER) & min t-DCF
Sama seperti pada baseline pertama, kita mendefinisikan perhitungan EER dengan sklearn/scipy serta min t-DCF.

In [ ]:
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def compute_eer_scipy(bonafide_scores, spoof_scores):
    """
    Menghitung Equal Error Rate (EER) menggunakan scipy.optimize.brentq dan sklearn.metrics.roc_curve.
    """
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    y_score = np.concatenate([bonafide_scores, spoof_scores])
    
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    
    eer = brentq(lambda x: interp1d(fpr, tpr)(x) + x - 1., 0., 1.)
    return eer

def compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores, Pfa_asv=0.22276, Pmiss_asv=0.06677, Pmiss_spoof_asv=0.76843):
    """
    Menghitung minimum Tandem Detection Cost Function (min t-DCF) standar ASVspoof 2019.
    """
    Pspoof = 0.05
    Ptar = (1.0 - Pspoof) * 0.99
    Pnon = (1.0 - Pspoof) * 0.01
    
    Cmiss_cm = 1.0
    Cfa_cm = 10.0
    Cmiss_asv = 1.0
    Cfa_asv = 10.0
    
    y_true = np.concatenate([np.ones_like(bonafide_scores), np.zeros_like(spoof_scores)])
    y_score = np.concatenate([bonafide_scores, spoof_scores])
    
    fpr, tpr, thresholds = roc_curve(y_true, y_score, pos_label=1)
    Pmiss_cm = 1.0 - tpr
    Pfa_cm = fpr
    
    C1 = Ptar * (Cmiss_cm - Cmiss_asv * Pmiss_asv) - Pnon * Cfa_asv * Pfa_asv
    C2 = Cfa_cm * Pspoof * (1.0 - Pmiss_spoof_asv)
    
    tDCF = C1 * Pmiss_cm + C2 * Pfa_cm
    tDCF_norm = tDCF / np.minimum(C1, C2)
    min_tdcf = np.min(tDCF_norm)
    return min_tdcf

print("Fungsi EER dan t-DCF siap digunakan.")

## 7. Training & Validation Loop
Di bawah ini mendefinisikan fungsi training epoch untuk model RawNet2. 
Menggunakan optimizer `Adam` (sesuai spesifikasi di file yaml model) dengan parameter `weight_decay = 0.0001` dan `lr = 0.0001`.

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

def train_rawnet_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # Reset gradien
        optimizer.zero_grad()
        
        # Forward pass (RawNet2 menghasilkan output logit berukuran [batch_size, 2])
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward & optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

def validate_rawnet(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    bonafide_scores = []
    spoof_scores = []
    
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            running_loss += loss.item() * batch_x.size(0)
            _, predicted = outputs.max(1)
            total += batch_y.size(0)
            correct += predicted.eq(batch_y).sum().item()
            
            # Gunakan probabilitas kelas bona fide (indeks 1) sebagai skor countermeasure
            probs = F.softmax(outputs, dim=1)
            bona_probs = probs[:, 1].cpu().numpy()
            labels_np = batch_y.cpu().numpy()
            
            for score, lbl in zip(bona_probs, labels_np):
                if lbl == 1:
                    bonafide_scores.append(score)
                else:
                    spoof_scores.append(score)
                    
    val_loss = running_loss / total
    val_acc = (correct / total) * 100
    
    bonafide_scores = np.array(bonafide_scores)
    spoof_scores = np.array(spoof_scores)
    
    if len(bonafide_scores) > 0 and len(spoof_scores) > 0:
        val_eer = compute_eer_scipy(bonafide_scores, spoof_scores) * 100
        val_tdcf = compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores)
    else:
        val_eer = 100.0
        val_tdcf = 1.0
        
    return val_loss, val_acc, val_eer, val_tdcf

print("Fungsi Train & Validate untuk RawNet2 berhasil dibuat.")

## 8. Eksekusi Pelatihan & Cross-Dataset Evaluation
Gunakan sel di bawah ini untuk menginisialisasi parameter pelatihan dan memulai training baseline RawNet2.

In [ ]:
# Tentukan device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# Inisialisasi model RawNet dengan parameter resmi
model = RawNet(rawnet2_config, device).to(device)

# Parameter Pelatihan
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0001)

# Pemuatan Dataloader (Menggunakan train_dataset yang didefinisikan di Bagian 2)
# Jika dataset lokal belum disiapkan, kode di bawah akan berjalan menggunakan dummy dataset.
train_dataset_raw = RawAudioASVDataset(
    protocols_file=PATH_PROTOKOL_TRAIN,
    wav_dir=PATH_WAV_DIR_TRAIN,
    max_len=64000
)
train_loader_raw = DataLoader(train_dataset_raw, batch_size=8, shuffle=True)

# Loop Pelatihan Baseline
num_epochs = 3 # Silakan sesuaikan dengan kebutuhan eksperimen Anda
print(f"\nMemulai Pelatihan RawNet2 selama {num_epochs} Epoch...\n")
print(f"{'Epoch':^8} | {'Train Loss':^12} | {'Train Acc (%)':^15}")
print("-" * 45)

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_rawnet_epoch(
        model=model,
        dataloader=train_loader_raw,
        optimizer=optimizer,
        criterion=criterion,
        device=device
    )
    
    print(f"{epoch:^8} | {train_loss:^12.4f} | {train_acc:^15.2f}")

print("\nPelatihan selesai!")

## 9. Cross-Dataset Evaluation (ASVspoof 5)
Evaluasi performa generalisasi model RawNet2 pada dataset ASVspoof 5.

In [ ]:
# --- Evaluasi Lintas-Dataset pada ASVspoof 5 ---
class ASVspoof5EvalDataset(Dataset):
    """
    Dataset PyTorch untuk memuat data evaluasi ASVspoof 5 (mendukung mode lokal & streaming HF).
    Mendukung struktur folder bersarang dengan pencarian path secara rekursif.
    """
    def __init__(self, samples, wav_dir, max_len=64000, use_local=True):
        self.samples = samples
        self.wav_dir = wav_dir
        self.max_len = max_len
        self.use_local = use_local
        
        # Bangun indeks path file audio secara rekursif jika menggunakan mode lokal
        self.path_map = {}
        if self.use_local and os.path.exists(wav_dir):
            print(f"Membangun indeks path audio secara rekursif di {wav_dir}...")
            for root, _, files in os.walk(wav_dir):
                for f in files:
                    if f.endswith(('.flac', '.wav')):
                        name_without_ext = os.path.splitext(f)[0]
                        self.path_map[name_without_ext] = os.path.join(root, f)
            print(f"Selesai! Berhasil mengindeks {len(self.path_map)} file audio.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        if self.use_local:
            file_name = item['file_name']
            label = item['label']
            
            # Cari path dari indeks map
            wav_path = self.path_map.get(file_name)
            
            # Jika tidak ditemukan di map, fallback ke os.path.join standar
            if not wav_path:
                wav_path = os.path.join(self.wav_dir, f"{file_name}.flac")
                if not os.path.exists(wav_path):
                    wav_path = os.path.join(self.wav_dir, f"{file_name}.wav")
                
            if wav_path and os.path.exists(wav_path):
                waveform, sr = torchaudio.load(wav_path)
                if sr != 16000:
                    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                    waveform = resampler(waveform)
                waveform = waveform.squeeze(0)
            else:
                waveform = torch.randn(self.max_len) * 0.01
        else:
            audio_data = item['audio']['array']
            sr = item['audio']['sampling_rate']
            waveform = torch.tensor(audio_data, dtype=torch.float32)
            
            if sr != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                waveform = resampler(waveform)
                
            if waveform.ndim > 1:
                waveform = waveform.mean(dim=0)
                
            label = item.get('label')
            if label is None:
                label = item.get('key') or item.get('class')
                
            is_bonafide = False
            if label in [1, '1', 'bonafide', 'bona_fide', 'human']:
                is_bonafide = True
            elif label in [0, '0', 'spoof', 'spoofed', 'attack']:
                is_bonafide = False
            else:
                is_bonafide = 'bonafide' in str(label).lower() or 'human' in str(label).lower()
            label = 1 if is_bonafide else 0
            
        if waveform.shape[0] < self.max_len:
            pad_len = self.max_len - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
        else:
            waveform = waveform[:self.max_len]
            
        return waveform, label
def evaluate_asvspoof5(model, eval_samples, wav_dir, device, use_local=True):
    print(f"\nMemulai evaluasi pada subset ASVspoof 5 terstratifikasi (mode_lokal={use_local})...")
    eval_dataset = ASVspoof5EvalDataset(eval_samples, wav_dir, max_len=64000, use_local=use_local)
    eval_loader = DataLoader(eval_dataset, batch_size=8, shuffle=False)
    
    model.eval()
    bonafide_scores = []
    spoof_scores = []
    
    with torch.no_grad():
        for batch_x, batch_y in eval_loader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            
            probs = F.softmax(outputs, dim=1)
            bona_probs = probs[:, 1].cpu().numpy()
            labels_np = batch_y.numpy()
            
            for score, lbl in zip(bona_probs, labels_np):
                if lbl == 1:
                    bonafide_scores.append(score)
                else:
                    spoof_scores.append(score)
                    
    bonafide_scores = np.array(bonafide_scores)
    spoof_scores = np.array(spoof_scores)
    
    if len(bonafide_scores) > 0 and len(spoof_scores) > 0:
        eer = compute_eer_scipy(bonafide_scores, spoof_scores) * 100
        min_tdcf = compute_asvspoof19_min_tdcf(bonafide_scores, spoof_scores)
        print(f"\n=========================================")
        print(f"Hasil Evaluasi Lintas-Dataset (ASVspoof 5):")
        print(f"EER      : {eer:.4f} %")
        print(f"min t-DCF: {min_tdcf:.6f}")
        print(f"=========================================")
    else:
        print("Gagal mengevaluasi: sampel kosong.")

if len(eval_samples) > 0:
    evaluate_asvspoof5(model, eval_samples, PATH_ASV5_WAV_DIR, device, use_local=use_local_asv5)
